# Advanced: Multi-Strategy Record Aggregation with PAMOLA.CORE

**Goal**: Master all 3 aggregation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Single-level grouping with multiple aggregations
- **Strategy 2**: Multi-level grouping (hierarchical aggregation)
- **Strategy 3**: Custom aggregation functions (weighted avg, percentiles)
- Calculate advanced privacy metrics (k-anonymity)
- Analyze data reduction and utility preservation
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/aggregate_records/
        └── 02_aggregate_records_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.grouping.aggregate_records_op import AggregateRecordsOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record sales dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 10 columns)
- Sample records (first 5 rows)
- Column statistics (5 regions, 5 categories, numeric amounts)

**Dataset features:**
- 1000 transaction records
- Multiple grouping dimensions (region, category, customer_segment)
- Numeric fields for aggregation (sale_amount, quantity, discount_percent)
- Boolean and categorical fields

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'aggregate_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Define dimensions for grouping
    regions = ['North', 'South', 'East', 'West', 'Central']
    categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Home']
    segments = ['Premium', 'Standard', 'Budget']
    
    df = pd.DataFrame({
        'transaction_id': range(1, n_records + 1),
        
        # Grouping fields
        'region': np.random.choice(regions, n_records),
        'product_category': np.random.choice(categories, n_records),
        'customer_segment': np.random.choice(segments, n_records),
        
        # Numeric fields for aggregation
        'sale_amount': np.random.uniform(10, 1000, n_records).round(2),
        'quantity': np.random.randint(1, 50, n_records),
        'discount_percent': np.random.uniform(0, 30, n_records).round(2),
        
        # Additional fields
        'customer_satisfaction': np.random.randint(1, 6, n_records),
        'is_loyalty_member': np.random.choice([True, False], n_records),
        'shipping_cost': np.random.uniform(5, 50, n_records).round(2),
        'processing_days': np.random.randint(1, 10, n_records),
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == 'object':
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_bool_dtype(df[col]):
        true_count = df[col].sum()
        false_count = (~df[col]).sum()
        print(f"  {col:25s} ({dtype_str:10s}): {true_count} True, {false_count} False")
    else:
        print(f"  {col:25s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}")

print(f"\n💡 Perfect dataset for testing all 3 aggregation strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit grouping and aggregation settings
   - `strategy1_group`: Single grouping field
   - `strategy1_aggs`: Aggregations for strategy 1
   - `strategy2_group`: Multi-level grouping fields
   - `strategy2_aggs`: Aggregations for strategy 2
   - `strategy3_group`: Grouping for custom aggregations
   - `strategy3_aggs`: Standard aggregations for strategy 3
2. Run to validate configuration and setup environment

**What you'll see:**
- Strategy 1: Single-level grouping validation
- Strategy 2: Multi-level grouping validation
- Strategy 3: Custom aggregation configuration
- Environment confirmations (✓ for each component)

**Note:** All grouping fields and aggregation fields validated before execution

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    # Strategy 1: Single-level grouping with multiple standard aggregations
    "strategy1_group": ["region"],
    "strategy1_aggs": {
        "sale_amount": ["sum", "mean", "count"],
        "quantity": ["sum", "mean"],
        "discount_percent": ["mean", "max"],
    },
    
    # Strategy 2: Multi-level grouping (hierarchical)
    "strategy2_group": ["region", "product_category"],
    "strategy2_aggs": {
        "sale_amount": ["sum", "mean"],
        "quantity": ["sum"],
        "customer_satisfaction": ["mean", "median"],
    },
    
    # Strategy 3: Custom aggregations with standard base
    "strategy3_group": ["customer_segment"],
    "strategy3_aggs": {
        "sale_amount": ["sum", "mean"],
        "quantity": ["sum"],
    },
    "strategy3_custom": {
        "sale_amount": ["percentile_90"],  # Percentiles
        "discount_percent": ["range_spread"],  # Custom: max - min
    },
}

# Validate Strategy 1
print("📋 Validating Strategy 1: Single-Level Grouping...\n")
for field_name in FIELD_CONFIG["strategy1_group"]:
    if field_name not in df.columns:
        raise ValueError(f"❌ Group field '{field_name}' not found!")
    print(f"  ✓ Group by '{field_name}': {df[field_name].nunique()} groups")

for agg_field in FIELD_CONFIG["strategy1_aggs"].keys():
    if agg_field not in df.columns:
        raise ValueError(f"❌ Aggregation field '{agg_field}' not found!")
    funcs = FIELD_CONFIG["strategy1_aggs"][agg_field]
    print(f"  ✓ '{agg_field}': {', '.join(funcs)}")

# Validate Strategy 2
print(f"\n📋 Validating Strategy 2: Multi-Level Grouping...\n")
for field_name in FIELD_CONFIG["strategy2_group"]:
    if field_name not in df.columns:
        raise ValueError(f"❌ Group field '{field_name}' not found!")
    print(f"  ✓ Group by '{field_name}': {df[field_name].nunique()} unique values")

# Calculate total groups for multi-level
total_groups_s2 = df.groupby(FIELD_CONFIG["strategy2_group"]).ngroups
print(f"  Total groups: {total_groups_s2}")

for agg_field in FIELD_CONFIG["strategy2_aggs"].keys():
    if agg_field not in df.columns:
        raise ValueError(f"❌ Aggregation field '{agg_field}' not found!")
    funcs = FIELD_CONFIG["strategy2_aggs"][agg_field]
    print(f"  ✓ '{agg_field}': {', '.join(funcs)}")

# Validate Strategy 3
print(f"\n📋 Validating Strategy 3: Custom Aggregations...\n")
for field_name in FIELD_CONFIG["strategy3_group"]:
    if field_name not in df.columns:
        raise ValueError(f"❌ Group field '{field_name}' not found!")
    print(f"  ✓ Group by '{field_name}': {df[field_name].nunique()} groups")

print(f"  Standard aggregations:")
for agg_field in FIELD_CONFIG["strategy3_aggs"].keys():
    if agg_field not in df.columns:
        raise ValueError(f"❌ Aggregation field '{agg_field}' not found!")
    funcs = FIELD_CONFIG["strategy3_aggs"][agg_field]
    print(f"    - '{agg_field}': {', '.join(funcs)}")

print(f"  Custom aggregations:")
for agg_field in FIELD_CONFIG["strategy3_custom"].keys():
    if agg_field not in df.columns:
        raise ValueError(f"❌ Aggregation field '{agg_field}' not found!")
    funcs = FIELD_CONFIG["strategy3_custom"][agg_field]
    print(f"    - '{agg_field}': {', '.join(funcs)}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_aggregation",
    description="Multi-strategy record aggregation processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Single-Level Grouping

**How to use:**
- Groups by single dimension (region)
- Applies multiple standard aggregations
- Run to execute strategy

**Key parameters:**
- `group_by_fields=["region"]`: Single grouping dimension
- `aggregations`: Multiple functions per field (sum, mean, count)
- `custom_aggregations=None`: No custom functions

**What you'll see:**
- Configuration summary
- Progress: validation → grouping → aggregation → metrics → viz → save
- Completion time (1-3 seconds expected)
- Record reduction (1000 → 5 groups)

**Note:** Best for simple dimension reduction with comprehensive statistics per group

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: SINGLE-LEVEL GROUPING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Single-level",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = AggregateRecordsOperation(
    # Core parameters
    group_by_fields=FIELD_CONFIG["strategy1_group"],
    aggregations=FIELD_CONFIG["strategy1_aggs"],
    custom_aggregations=None,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_dask=False,
    npartitions=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Group by: {FIELD_CONFIG['strategy1_group']}")
print(f"  Aggregations: {len(FIELD_CONFIG['strategy1_aggs'])} fields")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_single_level',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_single_level' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    print(f"\n📊 Reduction: {len(df)} → {len(result_df_s1)} groups")
    print(f"   Reduction ratio: {len(result_df_s1)/len(df):.4f} ({(1-len(result_df_s1)/len(df))*100:.1f}% reduction)")

## STRATEGY 2: Multi-Level Grouping

**How to use:**
- Groups by multiple dimensions (region + product_category)
- Creates hierarchical aggregation structure
- Run to execute strategy

**Key parameters:**
- `group_by_fields=["region", "product_category"]`: Two-level hierarchy
- `aggregations`: Standard functions (sum, mean, median)
- `custom_aggregations=None`: No custom functions

**What you'll see:**
- Configuration summary
- Progress: validation → multi-level grouping → aggregation → metrics → viz → save
- Completion time (1-3 seconds expected)
- Record reduction (1000 → 25 groups)

**Note:** Best for preserving more granularity with hierarchical group structure

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: MULTI-LEVEL GROUPING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Multi-level",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = AggregateRecordsOperation(
    # Core parameters
    group_by_fields=FIELD_CONFIG["strategy2_group"],
    aggregations=FIELD_CONFIG["strategy2_aggs"],
    custom_aggregations=None,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_dask=False,
    npartitions=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Group by: {FIELD_CONFIG['strategy2_group']}")
print(f"  Aggregations: {len(FIELD_CONFIG['strategy2_aggs'])} fields")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_multi_level',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_multi_level' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    print(f"\n📊 Reduction: {len(df)} → {len(result_df_s2)} groups")
    print(f"   Reduction ratio: {len(result_df_s2)/len(df):.4f} ({(1-len(result_df_s2)/len(df))*100:.1f}% reduction)")

## STRATEGY 3: Custom Aggregation Functions

**How to use:**
- Groups by customer segment
- Applies standard AND custom aggregations
- Run to execute strategy

**Key parameters:**
- `group_by_fields=["customer_segment"]`: Single dimension
- `aggregations`: Standard functions (sum, mean)
- `custom_aggregations`: Custom functions (percentile_90, range_spread...)

**Custom functions:**
- `range_spread`: Calculate the range (max - min).
- `percentile_90`: 90th percentile

**What you'll see:**
- Configuration summary with custom functions
- Progress: validation → grouping → custom aggregation → metrics → viz → save
- Completion time (1-3 seconds expected)
- Record reduction (1000 → 3 groups)

**Note:** Best for specialized statistical measures beyond standard aggregations

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: CUSTOM AGGREGATION FUNCTIONS")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Custom",
    unit="steps",
    track_memory=True,
    level=0
)

# Combine standard and custom aggregations
combined_aggs_s3 = FIELD_CONFIG["strategy3_aggs"].copy()
combined_custom_s3 = FIELD_CONFIG["strategy3_custom"].copy()

operation_s3 = AggregateRecordsOperation(
    # Core parameters
    group_by_fields=FIELD_CONFIG["strategy3_group"],
    aggregations=combined_aggs_s3,
    custom_aggregations=combined_custom_s3,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_dask=False,
    npartitions=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Group by: {FIELD_CONFIG['strategy3_group']}")
print(f"  Standard aggregations: {len(FIELD_CONFIG['strategy3_aggs'])} fields")
print(f"  Custom aggregations: {len(FIELD_CONFIG['strategy3_custom'])} fields")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_custom',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_custom' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    print(f"\n📊 Reduction: {len(df)} → {len(result_df_s3)} groups")
    print(f"   Reduction ratio: {len(result_df_s3)/len(df):.4f} ({(1-len(result_df_s3)/len(df))*100:.1f}% reduction)")

## Step 4: Strategy Comparison

**How to use:**
- Review execution times and reduction ratios
- Compare group counts across strategies
- Identify best approach for your use case

**What you'll see:**
- Execution time for each strategy
- Progressive group counts (5 → 25 → 3 groups)
- Reduction ratios (smaller = better privacy)
- Total processing time

**Note:** More groups = higher utility, fewer groups = better privacy

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Single-level):  {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Multi-level):   {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Custom):        {elapsed_s3:6.2f}s")
print(f"  Total:                      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Record Reduction:")
print(f"  Original records:           {len(df):4d}")
print(f"  Strategy 1 groups:          {len(result_df_s1):4d} ({len(result_df_s1)/len(df):.4f} ratio, {(1-len(result_df_s1)/len(df))*100:5.1f}% reduction)")
print(f"  Strategy 2 groups:          {len(result_df_s2):4d} ({len(result_df_s2)/len(df):.4f} ratio, {(1-len(result_df_s2)/len(df))*100:5.1f}% reduction)")
print(f"  Strategy 3 groups:          {len(result_df_s3):4d} ({len(result_df_s3)/len(df):.4f} ratio, {(1-len(result_df_s3)/len(df))*100:5.1f}% reduction)")

print(f"\n📈 Aggregated Field Counts:")
print(f"  Strategy 1: {len(result_df_s1.columns)} columns")
print(f"  Strategy 2: {len(result_df_s2.columns)} columns")
print(f"  Strategy 3: {len(result_df_s3.columns)} columns")

print(f"\n💡 Strategy Selection:")
print(f"  - Strategy 1: Best for simple dimension reduction")
print(f"  - Strategy 2: Best balance of utility and privacy")
print(f"  - Strategy 3: Maximum privacy with custom statistics")

## Step 5: Detailed Artifact Review

**How to use:**
- Review all generated artifacts from each strategy
- Auto-loads NEWEST files from each category
- Displays metrics and visualizations inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Group counts, reduction ratios, aggregation statistics
2. **Visualizations**: Group distributions and aggregation results (first 2 displayed)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_single_level', 'Strategy 1: Single-Level Grouping'),
    ('strategy2_multi_level', 'Strategy 2: Multi-Level Grouping'),
    ('strategy3_custom', 'Strategy 3: Custom Aggregations')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                print(f"   Input records: {metrics.get('total_input_records', 0)}")
                print(f"   Output records: {metrics.get('total_output_records', 0)}")
                print(f"   Number of groups: {metrics.get('num_groups', 0)}")
                print(f"   Reduction ratio: {metrics.get('reduction_ratio', 0):.4f}")
                print(f"   Execution: {metrics.get('execution_time_seconds', 0):.4f}s")
                
                # Group size stats
                print(f"\n   Group size statistics:")
                print(f"     Min: {metrics.get('group_size_min', 0)}")
                print(f"     Max: {metrics.get('group_size_max', 0)}")
                print(f"     Mean: {metrics.get('group_size_mean', 0):.2f}")
                print(f"     Median: {metrics.get('group_size_median', 0):.2f}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Calculate k-anonymity for each strategy
- Compare privacy improvements
- Run AFTER all strategies complete

**What you'll see:**
- Original data: records per quasi-identifier combination
- Per strategy: min group size (k-anonymity level)
- Comparison: which strategy provides best privacy

**Privacy targets:**
- Min group size ≥ 5: Basic protection
- Min group size ≥ 10: Strong protection
- Larger groups: Better privacy

**Note:** Group size = k-anonymity level (higher is better for privacy)

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

# Strategy 1: Single-level grouping
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("📊 STRATEGY 1 (Single-Level):")
    
    # Calculate group sizes from original data
    group_sizes_s1 = df.groupby(FIELD_CONFIG['strategy1_group']).size()
    
    print(f"  Number of groups: {len(group_sizes_s1)}")
    print(f"  Min group size: {group_sizes_s1.min()} (k-anonymity level)")
    print(f"  Max group size: {group_sizes_s1.max()}")
    print(f"  Mean group size: {group_sizes_s1.mean():.2f}")
    print(f"  Median group size: {group_sizes_s1.median():.2f}")
    print(f"\n  Privacy assessment: {group_sizes_s1.min()}-anonymous")

# Strategy 2: Multi-level grouping
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print(f"\n📊 STRATEGY 2 (Multi-Level):")
    
    # Calculate group sizes from original data
    group_sizes_s2 = df.groupby(FIELD_CONFIG['strategy2_group']).size()
    
    print(f"  Number of groups: {len(group_sizes_s2)}")
    print(f"  Min group size: {group_sizes_s2.min()} (k-anonymity level)")
    print(f"  Max group size: {group_sizes_s2.max()}")
    print(f"  Mean group size: {group_sizes_s2.mean():.2f}")
    print(f"  Median group size: {group_sizes_s2.median():.2f}")
    print(f"\n  Privacy assessment: {group_sizes_s2.min()}-anonymous")

# Strategy 3: Custom aggregations
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print(f"\n📊 STRATEGY 3 (Custom):")
    
    # Calculate group sizes from original data
    group_sizes_s3 = df.groupby(FIELD_CONFIG['strategy3_group']).size()
    
    print(f"  Number of groups: {len(group_sizes_s3)}")
    print(f"  Min group size: {group_sizes_s3.min()} (k-anonymity level)")
    print(f"  Max group size: {group_sizes_s3.max()}")
    print(f"  Mean group size: {group_sizes_s3.mean():.2f}")
    print(f"  Median group size: {group_sizes_s3.median():.2f}")
    print(f"\n  Privacy assessment: {group_sizes_s3.min()}-anonymous")

# Comparison
print(f"\n✨ PRIVACY COMPARISON:")
if all(var in locals() for var in ['group_sizes_s1', 'group_sizes_s2', 'group_sizes_s3']):
    k_values = {
        'Strategy 1': group_sizes_s1.min(),
        'Strategy 2': group_sizes_s2.min(),
        'Strategy 3': group_sizes_s3.min()
    }
    
    best_strategy = max(k_values, key=k_values.get)
    best_k = k_values[best_strategy]
    
    print(f"  Best privacy: {best_strategy} ({best_k}-anonymous)")
    print(f"\n  All strategies achieve minimum k-anonymity of:")
    for strategy, k_val in sorted(k_values.items(), key=lambda x: x[1], reverse=True):
        print(f"    {strategy}: {k_val}")

## Step 7: Export Final Data

**How to use:**
- Export final aggregated datasets from each strategy
- Each strategy saves to its own folder
- Run AFTER all strategies complete

**What you'll see (per strategy):**
- Available columns list
- Export confirmation (path, rows, columns)
- Preview of first 5 rows
- Group summary

**Output structure:**
```
advanced_output/
├── strategy1_single_level/aggregated_data.csv
├── strategy2_multi_level/aggregated_data.csv
└── strategy3_custom/aggregated_data.csv
```

**Note:** Progressive aggregation visible across strategy outputs

In [ ]:
print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Single-Level Grouping
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: SINGLE-LEVEL GROUPING")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_single_level'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s1 = s1_dir / 'aggregated_data.csv'
    try:
        result_df_s1.to_csv(output_path_s1, index=False)
        print(f"\n✅ Saved: {output_path_s1}")
        print(f"   Rows: {len(result_df_s1):,}")
        print(f"   Columns: {len(result_df_s1.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s1.head())
    
    print(f"\n📈 Groups: {len(result_df_s1)}")
    print(f"   Grouping fields: {', '.join(FIELD_CONFIG['strategy1_group'])}")

# ============================================================================
# STRATEGY 2: Multi-Level Grouping
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: MULTI-LEVEL GROUPING")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_multi_level'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s2 = s2_dir / 'aggregated_data.csv'
    try:
        result_df_s2.to_csv(output_path_s2, index=False)
        print(f"\n✅ Saved: {output_path_s2}")
        print(f"   Rows: {len(result_df_s2):,}")
        print(f"   Columns: {len(result_df_s2.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s2.head())
    
    print(f"\n📈 Groups: {len(result_df_s2)}")
    print(f"   Grouping fields: {', '.join(FIELD_CONFIG['strategy2_group'])}")

# ============================================================================
# STRATEGY 3: Custom Aggregations
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: CUSTOM AGGREGATIONS")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_custom'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s3 = s3_dir / 'aggregated_data.csv'
    try:
        result_df_s3.to_csv(output_path_s3, index=False)
        print(f"\n✅ Saved: {output_path_s3}")
        print(f"   Rows: {len(result_df_s3):,}")
        print(f"   Columns: {len(result_df_s3.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s3.head())
    
    print(f"\n📈 Groups: {len(result_df_s3)}")
    print(f"   Grouping fields: {', '.join(FIELD_CONFIG['strategy3_group'])}")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 aggregation strategies implemented and compared
- ✅ Privacy metrics calculated (k-anonymity)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Single-level: Maximum privacy (fewest groups, largest k-anonymity)
- Multi-level: Balance of utility and privacy (hierarchical structure)
- Custom: Specialized statistics while maintaining privacy
- All strategies preserve statistical utility through aggregation

**Strategy recommendations:**
- **Use Strategy 1** when: Maximum privacy required, simple dimension reduction
- **Use Strategy 2** when: Need hierarchical analysis, preserve more granularity
- **Use Strategy 3** when: Specialized statistical measures needed (percentiles, ranges)

**Performance insights:**
- All strategies execute in 1-3 seconds on 1000 records
- Custom aggregations add minimal overhead
- Multi-level grouping preserves more information

**Next steps:**
- Test with your own datasets
- Define custom aggregation functions for your domain
- Combine with other privacy operations
- Deploy to production environment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)